# Imports

In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

In [2]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Start

In [ ]:
data_url = 'https://raw.githubusercontent.com/Hospital-Da-Luz-Learning-Health/IMLHCatolica26/main/Aula%205%20-%20Supervised%20Learning%20II/data/liver_disease.csv'
df = pd.read_csv(data_url, index_col = 'id')
df.head(3)
df.shape

,Age,Gender,BMI,AlcoholConsumption,Smoking,GeneticRisk,PhysicalActivity,Diabetes,Hypertension,LiverFunctionTest,Diagnosis
id,,,,,,,,,,,
572,75,1,32.960731,10.911642,0,0,8.930014,0,1,89.950963,1
125,20,1,30.298513,9.417347,0,0,1.484017,0,0,75.777573,1
1553,21,1,33.230869,1.685287,1,1,3.411027,0,0,69.226081,1


(804, 11)

In [4]:
X = df.drop(columns='Diagnosis')
y = df['Diagnosis']

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.35, random_state=42, stratify=y)

# Decision Tree

Lookup the parameters in the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html). Or ask chatgpt to help you build a parameter grid for logistic regression from sklearn

In [6]:
from sklearn.tree import DecisionTreeClassifier

In [7]:
param_grid = {
    'max_depth': [3, 5, 7, 9],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2']
}

In [8]:
grid = GridSearchCV(DecisionTreeClassifier(), # your model
                    param_grid, #the parameter grid
                    cv=3, #how many folds do you want in your cross-validation
                    scoring='f1', # what scoring metric do you want. More here: https://scikit-learn.org/stable/modules/model_evaluation.html
                    verbose=1, # to show the messages during training
                    n_jobs=-1 # how many cores to use in your computer
                    )

In [9]:
grid.fit(X_train, y_train)

Fitting 3 folds for each of 108 candidates, totalling 324 fits


,estimator,DecisionTreeClassifier()
,param_grid,"{'max_depth': [3, 5, ...], 'max_features': [None, 'sqrt', ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...]}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,3
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'gini'


In [10]:
results = pd.DataFrame(grid.cv_results_).sort_values('rank_test_score')
results.head(5)
results.shape

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_max_depth,param_max_features,param_min_samples_leaf,param_min_samples_split,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score
77,0.052676,0.029732,0.053537,0.043513,7,log2,2,10,"{'max_depth': 7, 'max_features': 'log2', 'min_...",0.285714,0.555556,0.266667,0.369312,0.131923,1
78,0.098701,0.078385,0.092415,0.031369,7,log2,4,2,"{'max_depth': 7, 'max_features': 'log2', 'min_...",0.166667,0.526316,0.363636,0.352206,0.147048,2
50,0.022589,0.011231,0.093260,0.011305,5,log2,2,10,"{'max_depth': 5, 'max_features': 'log2', 'min_...",0.181818,0.555556,0.307692,0.348355,0.155263,3
59,0.068983,0.061936,0.068042,0.040831,7,None,2,10,"{'max_depth': 7, 'max_features': None, 'min_sa...",0.153846,0.555556,0.333333,0.347578,0.164306,4
58,0.068668,0.069031,0.147517,0.067638,7,None,2,5,"{'max_depth': 7, 'max_features': None, 'min_sa...",0.142857,0.526316,0.333333,0.334169,0.156547,5


(108, 15)

# Picking the best model

In [11]:
best_model = grid.best_estimator_

Using that model to estimate performance on a test set

In [17]:
y_pred = best_model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.98      0.97       268
           1       0.45      0.36      0.40        14

    accuracy                           0.95       282
   macro avg       0.71      0.67      0.69       282
weighted avg       0.94      0.95      0.94       282



# RandomSearch

When your dataset is too large, searching a very large hyperparameter space becomes unfeasable

In [18]:
param_grid = {
    'max_depth': [3, 5, 7, 9],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2']
}

In [ ]:
# init random search
rand = RandomizedSearchCV(DecisionTreeClassifier(), # your model
                    param_grid, #the parameter grid
                    cv=3, #how many folds do you want in your cross-validation
                    scoring='f1', # what scoring metric do you want. More here: https://scikit-learn.org/stable/modules/model_evaluation.html
                    verbose=1, # to show the messages during training
                    n_jobs=-1, # how many cores to use in your computer
                    n_iter=20, # how many iterations do you want to do,
                    random_state=40
                    )

In [ ]:
rand.fit(X_train, y_train)

Fitting 3 folds for each of 20 candidates, totalling 60 fits


,estimator,DecisionTreeClassifier()
,param_distributions,"{'max_depth': [3, 5, ...], 'max_features': [None, 'sqrt', ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...]}"
,n_iter,20
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,3
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,40
,error_score,nan


In [ ]:
results = pd.DataFrame(rand.cv_results_).sort_values('rank_test_score')
results.head(5)
results.shape

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_min_samples_split,param_min_samples_leaf,param_max_features,param_max_depth,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score
6,0.046165,0.047130,0.072685,0.046711,5,2,log2,9,"{'min_samples_split': 5, 'min_samples_leaf': 2...",0.181818,0.500000,0.166667,0.282828,0.153688,1
18,0.014970,0.004824,0.024467,0.005616,10,4,None,9,"{'min_samples_split': 10, 'min_samples_leaf': ...",0.000000,0.476190,0.333333,0.269841,0.199521,2
16,0.009850,0.002909,0.110135,0.068474,2,4,sqrt,5,"{'min_samples_split': 2, 'min_samples_leaf': 4...",0.307692,0.285714,0.181818,0.258408,0.054896,3
8,0.049099,0.027769,0.020068,0.002948,5,2,None,3,"{'min_samples_split': 5, 'min_samples_leaf': 2...",0.222222,0.285714,0.266667,0.258201,0.026603,4
10,0.010225,0.001547,0.040660,0.026776,10,2,None,5,"{'min_samples_split': 10, 'min_samples_leaf': ...",0.153846,0.526316,0.000000,0.226721,0.220960,5


(20, 15)

In [ ]:
best_model = rand.best_estimator_

Using that model to estimate performance on a test set

In [23]:
y_pred = best_model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      1.00      0.98       268
           1       0.67      0.14      0.24        14

    accuracy                           0.95       282
   macro avg       0.81      0.57      0.61       282
weighted avg       0.94      0.95      0.94       282

